# Attention Weight Structure

Properties of attention weight matrices: QK/OV norms, rank analysis,
QK-OV alignment, and summary.

## Why This Matters

Attention weight structure reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_weight_structure import (
    qk_weight_norms, ov_weight_norms,
    weight_matrix_rank_analysis, qk_ov_alignment,
    weight_structure_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print('Model ready')

## QK Weight Norms

Query and key projection strengths per head.

In [ ]:
result = qk_weight_norms(model)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: Q={p['mean_q_norm']:.4f}, K={p['mean_k_norm']:.4f}")
    for h in p['per_head']:
        print(f"    H{h['head']}: Q={h['q_norm']:.4f}, K={h['k_norm']:.4f}, ratio={h['qk_ratio']:.4f}")

## OV Weight Norms

Value and output projection strengths.

In [ ]:
result = ov_weight_norms(model)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: V={p['mean_v_norm']:.4f}, O={p['mean_o_norm']:.4f}")
    for h in p['per_head']:
        print(f"    H{h['head']}: V={h['v_norm']:.4f}, O={h['o_norm']:.4f}, OV={h['ov_product_norm']:.4f}")

## Weight Matrix Rank Analysis

Effective rank and condition number of weight matrices.

In [ ]:
for layer in range(4):
    for head in range(4):
        r = weight_matrix_rank_analysis(model, layer, head)
        ranks = {k: f"{v['effective_rank']:.2f}" for k, v in r['matrices'].items()}
        print(f"  L{layer}H{head}: {ranks}")

## QK-OV Alignment

Do the QK (matching) and OV (copying) circuits align?

In [ ]:
for layer in range(4):
    for head in range(4):
        r = qk_ov_alignment(model, layer, head)
        flag = ' [ALIGNED]' if r['is_aligned'] else ''
        print(f"  L{layer}H{head}: align={r['qk_ov_alignment']:.4f}{flag}")

## Weight Structure Summary

In [ ]:
result = weight_structure_summary(model)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: Q={p['mean_q_norm']:.4f}, K={p['mean_k_norm']:.4f}, "
          f"V={p['mean_v_norm']:.4f}, O={p['mean_o_norm']:.4f}")